# 🚗 Truck Blind Spot Detection - Notebook Demo

Notebook này hướng dẫn cách sử dụng `BlindSpotPipeline` để phát hiện vật thể trong vùng điểm mù trực tiếp từ code Python.

--- 
### 🎁 Cách tận dụng Google Colab Pro (H100/A100)
1. Truy cập **Runtime** -> **Change runtime type** -> Chọn GPU cao nhất (Vd: **H100** hoặc **A100**).
2. Chạy cell **Setup for Google Colab** để cài đặt môi trường.
3. Code bên dưới sẽ tự động phát hiện GPU và khởi tạo model YOLOv9 trên GPU để đạt FPS tối đa (~30-50 FPS).

### 0. Setup for Google Colab (Chỉ chạy nếu dùng Colab)

In [4]:
import os
import sys

# 1. Clone project từ GitHub (nhánh taitu)
repo_url = "https://github.com/VTD0102/truck_blind_spot.git"
branch = "taitu"
project_dir = "/content/truck_blind_spot"

# Nếu đã clone rồi thì bỏ qua, không clone lại
if not os.path.exists(project_dir):
    !git clone --branch {branch} --single-branch {repo_url}
else:
    print(f"✅ Repo đã tồn tại ở {project_dir}")

# 2. Đi vào thư mục project
%cd /content/truck_blind_spot

# 3. Cài dependencies
if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt --quiet
    print("✅ Đã cài requirements xong!")
else:
    print("⚠️ Không tìm thấy requirements.txt trong thư mục project hiện tại.")
    !pwd
    !ls -la

Cloning into 'truck_blind_spot'...
remote: Enumerating objects: 6479, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 6479 (delta 11), reused 15 (delta 6), pack-reused 6445 (from 1)
Receiving objects: 100% (6479/6479), 267.57 MiB | 33.56 MiB/s, done.
Resolving deltas: 100% (1873/1873), done.
/content/truck_blind_spot
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.2 MB/s eta 0:00:0000:01
✅ Đã cài requirements xong!


### 1. Import dependencies & Check GPU

In [7]:
import os
import sys
import cv2
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image, clear_output

project_root = "/content/truck_blind_spot"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.pipeline import BlindSpotPipeline

%matplotlib inline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"📌 Device: {device}")
if device == "cuda:0":
    print(f"🔥 GPU Model: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'src'

### 2. Khởi tạo Pipeline

In [5]:
from pathlib import Path

weights_path = "weights/best_small.pt"
roi_config_path = "configs/roi.json"
classes_config_path = "configs/classes.yaml"

assert Path(weights_path).exists(), f"Không tìm thấy {weights_path}"
assert Path(roi_config_path).exists(), f"Không tìm thấy {roi_config_path}"
assert Path(classes_config_path).exists(), f"Không tìm thấy {classes_config_path}"

pipeline = BlindSpotPipeline(
    weights_path=weights_path,
    roi_config_path=roi_config_path,
    classes_config_path=classes_config_path,
    device=device,
    conf_threshold=0.25,
    iou_threshold=0.45
)

print(f"✅ Pipeline initialized on {device}!")

YOLO 🚀 2026-4-3 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)

Fusing layers... 
yolov9-s summary: 658 layers, 9600344 parameters, 0 gradients, 38.7 GFLOPs


✅ Pipeline initialized on cuda:0!


### 3. Chạy video demo (Hiển thị trong Notebook)

In [1]:
video_path = "assets/videos/demo.mp4"
cap = cv2.VideoCapture(video_path)

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        annotated_frame, _ = pipeline.process_frame(frame)

        # Encode JPEG để hiển thị trong Colab
        success_enc, buffer = cv2.imencode(".jpg", annotated_frame)
        if not success_enc:
            continue

        clear_output(wait=True)
        display(Image(data=buffer.tobytes()))

finally:
    cap.release()
    print("🎬 Video processed.")

NameError: name 'cv2' is not defined